# S1 — NumPy 與向量化思維（解答版 / Answer Key）

> ⚠️ **警告（給學員）**：請先在 `S1_numpy_vectorization.ipynb` 自行完成三題練習後再打開本檔！
> 直接看答案會大幅降低學習效果。向量化思維是「動手卡關」才會內化的肌肉記憶。

---

## 本檔定位

- **對象**：授課教師、助教、自學者完成練習後的對照
- **內容**：S1 課堂練習三題（送分 / 核心 / 挑戰）的完整解答 + 背後原因
- **重點**：不是只給「可跑的程式碼」，而是解釋**為什麼要這樣寫**，以及新手常踩的坑
- **資料**：`../datasets/ecommerce/products.csv`（與 S1 相同，30 個商品）

---

## 環境與資料載入

與 S1 實務 Case 相同，使用 `np.genfromtxt` 直接讀 CSV 的數值欄位。
之所以「只讀數值欄」，是因為本節尚未教 Pandas，NumPy 的 ndarray 對混合型別（字串＋數字）支援不佳。

In [1]:
import numpy as np

# 與 S1 同一份資料來源，路徑相對於 notebook 所在目錄
DATA = '../datasets/ecommerce/products.csv'

# usecols=3 是 unit_price 欄位（第 4 欄，0-indexed）
# usecols=4 是 stock_qty 欄位（第 5 欄）
# skip_header=1 是為了跳過標題列，否則會被當成 NaN
prices = np.genfromtxt(DATA, delimiter=',', skip_header=1, usecols=3)
stocks = np.genfromtxt(DATA, delimiter=',', skip_header=1, usecols=4)

print('商品數 =', len(prices))
print('前 5 筆價格:', prices[:5])
print('前 5 筆庫存:', stocks[:5])

商品數 = 30
前 5 筆價格: [1627. 1875.  620. 1718.  290.]
前 5 筆庫存: [ 12. 125.  52. 279. 216.]


---
## 🟢 送分題 — 解答

**題目**：建立一個 `np.array`，內容是 `[10, 20, 30, 40, 50]`，計算：
1. 所有元素的平均
2. 所有元素乘以 2 之後的結果
3. 篩出大於 25 的元素

**教學目的**：讓學員第一次親手用到 `.mean()`、廣播、布林遮罩三大基本功。

In [2]:
# Step 0: 建立陣列
# 為什麼用 np.array 而不是 Python list？
# 因為後面要做的「整批運算」list 做不到（list * 2 會變成重複兩次，不是每個元素乘 2）
a = np.array([10, 20, 30, 40, 50])
print('a =', a)

# 小題 1：平均
# .mean() 是 ndarray 的 method，內部是 C 實作，比 sum(a)/len(a) 快且更標準
mean_val = a.mean()
print(f'1) 平均 = {mean_val}')  # (10+20+30+40+50)/5 = 30.0

# 小題 2：每個元素乘以 2（廣播 broadcasting）
# a * 2 NumPy 會把「純量 2」自動擴展成與 a 同形狀再相乘
# 這就是向量化：一行取代 for-loop
doubled = a * 2
print(f'2) 乘以 2 = {doubled}')  # [20 40 60 80 100]

# 小題 3：篩出 > 25 的元素（布林遮罩 boolean mask）
# a > 25 產生 [F F T T T]，再用它當索引拿回 True 位置的值
# 常見錯誤：寫成 a[a>25 and a<100]（用 and 會 ValueError），
#           正確要用 & 並加括號：a[(a>25) & (a<100)]
filtered = a[a > 25]
print(f'3) 大於 25 = {filtered}')  # [30 40 50]

a = [10 20 30 40 50]
1) 平均 = 30.0
2) 乘以 2 = [ 20  40  60  80 100]
3) 大於 25 = [30 40 50]


### 💡 送分題 — 思考延伸

- `.mean()` vs `np.mean(a)`：兩者等價，前者是 method 呼叫、後者是 function 呼叫。教學上建議用 method（物件導向風格）。
- 為什麼 `a * 2` 不會修改原本的 `a`？因為 NumPy 運算預設**回傳新陣列**，符合 immutability 原則，這對後續 debug 非常重要。

> ### 💡 知識補給站 — Why ndarray is 100x Faster than list
> 
> Python list 是一個「**指標陣列**」— 每個元素可能散落在記憶體的不同位置，CPU 每次都要跳到新地址去找資料。
> 
> ndarray 是一塊「**連續記憶體 (contiguous C-array)**」— 所有元素緊密排列在一起，CPU 可以像翻書一樣從頭讀到尾，配合 **CPU cache** 極度高效。
> 
> 類比：list 像把 100 張發票塞在不同抽屜，每張都要開抽屜找；ndarray 像把 100 張發票排在同一本資料夾裡，翻頁就好。
> 
> 這也是為什麼 ndarray 要求「**同型別 (homogeneous dtype)**」— 每格大小一致才能連續排列，這是速度的代價。
> 
> *延伸關鍵字：memory layout, CPU cache locality, contiguous array, homogeneous dtype*

---
## 🟡 核心題 — 解答

**題目**：使用 `products.csv`，算出：
1. 單價 > 1000 的商品總共有幾個？
2. 庫存最多的前 3 個商品的**索引位置**（提示：`np.argsort`）
3. 假設「單價 < 500 的商品」全部進貨 50 個，這些商品總共要花多少錢？

**教學目的**：把前面三個觀念（遮罩、統計、索引）組合起來解**真實商業問題**。

In [3]:
# 小題 1：單價 > 1000 的商品數
# 關鍵技巧：布林陣列的 .sum() 會把 True 當 1、False 當 0 相加
# 這就是「計數符合條件的元素」最簡潔的寫法
mask_expensive = prices > 1000
num_expensive = mask_expensive.sum()
print(f'1) 單價 > 1000 的商品數 = {num_expensive} 個')

# 常見錯誤：用 len(prices[prices>1000]) —— 可以跑但多一次陣列複製，沒必要
# 更糟的：用 for-loop 自己數 counter += 1，違反向量化原則

1) 單價 > 1000 的商品數 = 16 個


In [4]:
# 小題 2：庫存最多的前 3 個商品的「索引位置」
# np.argsort 回傳的是「排序後的索引」而非值本身（這是新手最容易誤解的）
# 預設是「升冪」→ 小的在前；要取最大的 3 個就要取尾巴 [-3:]
# 再用 [::-1] 反轉，讓「最大」排在第一個（符合 top-3 語意）
sorted_idx = np.argsort(stocks)          # 升冪索引
top3_idx = sorted_idx[-3:][::-1]         # 取最後三個並反轉成降冪
print(f'2) 庫存前 3 名索引 = {top3_idx}')
print(f'   對應庫存量     = {stocks[top3_idx]}')

# 另一種等價寫法（進階）：np.argsort(-stocks)[:3]
# 原理：對負值升冪 ≡ 對原值降冪，適合有負號運算的情境

2) 庫存前 3 名索引 = [7 3 8]
   對應庫存量     = [287. 279. 279.]


In [5]:
# 小題 3：單價 < 500 的商品，每個進貨 50 個，總共多少錢？
# 拆解：(a) 篩出符合條件的價格 → (b) 每個乘 50 → (c) 加總
# 這三步完全不需要 for-loop
mask_cheap = prices < 500
cheap_prices = prices[mask_cheap]        # 篩出符合條件的單價
total_cost = (cheap_prices * 50).sum()   # 每項進 50 個，總成本

print(f'3) 便宜商品數量 = {mask_cheap.sum()} 項')
print(f'   總進貨成本   = NT$ {total_cost:,.0f}')

# 更精簡的一行寫法（教學建議先拆解再展示）：
# total_cost = (prices[prices < 500] * 50).sum()

3) 便宜商品數量 = 8 項
   總進貨成本   = NT$ 120,300


### 💡 核心題 — 思考延伸

- **布林遮罩的威力**：同一個 mask 可以套在不同陣列（例如 `stocks[mask_cheap]` 取對應的庫存），這在多欄位聯動分析時極有用。
- **argsort 的陷阱**：學員常誤把它回傳值當成「排序後的陣列」。正確流程是 `argsort → 索引 → 代回原陣列`。
- **為什麼不直接 `np.sort(stocks)[-3:]`？** 因為題目要「索引位置」而非值；索引位置才能對應回 product_id。

> ### 💡 知識補給站 — Broadcasting 三條規則
> 
> 廣播不是魔法，NumPy 遵守三條嚴格規則（從**最後一個維度**開始比對）：
> 
> 1. 如果兩個陣列的維度數不同，較少維度的那個在**前面補 1**
> 2. 兩個維度的 size 必須**相同**，或其中一個是 **1**
> 3. Size 為 1 的那個維度會被「拉長」到對方的 size
> 
> 類比：Excel 公式 `=A1:A100 * B1` → B1 就是被「廣播」到 100 列。如果規則不符合，NumPy 會報 `ValueError: operands could not be broadcast together`。
> 
> *延伸關鍵字：broadcasting rules, shape compatibility, dimension alignment*

In [ ]:
# 💡 知識補給站 demo — Broadcasting 視覺化
a = np.arange(3).reshape(3, 1)   # shape (3,1)
b = np.arange(4).reshape(1, 4)   # shape (1,4)
print(f'a shape: {a.shape}, b shape: {b.shape}')
print(f'a * b shape: {(a * b).shape}')   # → (3, 4)
print(a * b)

---
## 🔴 挑戰題 — 解答

**題目**：雙 11 分級折扣
- 庫存 ≥ 100：打 7 折
- 庫存 20~99：打 9 折
- 庫存 < 20：原價（快缺貨不打折）

請**完全向量化**（禁止 for-loop）算出每個商品的雙 11 售價。

**教學目的**：學會用 `np.where` 處理「多條件分支」——這是從 Excel 思維轉換成資料科學思維的關鍵跳板。

In [6]:
# np.where(condition, value_if_true, value_if_false)
# 像是向量化版的 if-else，但一次處理整個陣列
#
# 三個分支要用「巢狀 where」拆成兩層：
#   最外層判斷「庫存 >= 100？」
#     True → 打 7 折
#     False → 再判斷「庫存 >= 20？」
#               True → 打 9 折
#               False → 原價
#
# 訣竅：分支條件要「互斥且完整」，否則會漏掉某區間

final_price = np.where(
    stocks >= 100,                       # 條件 1：超量庫存
    prices * 0.7,                        # → 7 折
    np.where(
        stocks >= 20,                    # 條件 2：中量庫存
        prices * 0.9,                    # → 9 折
        prices                           # 否則：原價（< 20）
    )
)

# 驗證每個區間的結果
print('庫存分佈:')
print(f'  >= 100 : {(stocks >= 100).sum()} 件（7 折）')
print(f'  20~99  : {((stocks >= 20) & (stocks < 100)).sum()} 件（9 折）')
print(f'  < 20   : {(stocks < 20).sum()} 件（原價）')

print('\n前 10 筆（原價 → 雙 11 售價 / 庫存）:')
for i in range(10):
    print(f'  [{i:2d}] {prices[i]:7.0f} → {final_price[i]:7.1f}  (stock={int(stocks[i])})')

print(f'\n全站雙 11 售價總和 = NT$ {final_price.sum():,.0f}')
print(f'全站原價總和       = NT$ {prices.sum():,.0f}')
print(f'預估折讓金額       = NT$ {(prices - final_price).sum():,.0f}')

庫存分佈:
  >= 100 : 21 件（7 折）
  20~99  : 7 件（9 折）
  < 20   : 2 件（原價）

前 10 筆（原價 → 雙 11 售價 / 庫存）:
  [ 0]    1627 →  1627.0  (stock=12)
  [ 1]    1875 →  1312.5  (stock=125)
  [ 2]     620 →   558.0  (stock=52)
  [ 3]    1718 →  1202.6  (stock=279)
  [ 4]     290 →   203.0  (stock=216)
  [ 5]     157 →   141.3  (stock=47)
  [ 6]     609 →   426.3  (stock=258)
  [ 7]    1537 →  1075.9  (stock=287)
  [ 8]     561 →   392.7  (stock=279)
  [ 9]    1095 →   766.5  (stock=229)

全站雙 11 售價總和 = NT$ 27,153
全站原價總和       = NT$ 34,644
預估折讓金額       = NT$ 7,491


### 💡 挑戰題 — 為什麼不能 for-loop？

一份真實電商常有**百萬筆 SKU**，for-loop 版本在 Python 中會慢上百倍。`np.where` 底層是 C 實作，能完全發揮 CPU 的 SIMD 指令。這不只是「寫得漂亮」，是**效能差距一個數量級以上**的實戰差別。

### 💡 等價替代方案（進階）

若條件更多（例如 5~6 個分支），用 `np.select` 會比巢狀 `np.where` 更清楚：

```python
conditions = [stocks >= 100, stocks >= 20]
choices    = [prices * 0.7, prices * 0.9]
final_price = np.select(conditions, choices, default=prices)
```

`np.select` 會依序檢查每個條件，第一個為 True 的就採用對應 choice，其餘走 default。

---
## 🎯 教學提示（給講師 / 助教）

以下彙整三題學員最常踩的坑，批改作業時可優先檢查。

### 送分題常見錯誤

1. **用 Python list 做**：`a = [10,20,30,40,50]; a*2` 會變 `[10,...,50,10,...,50]`（重複兩次），而非逐元素乘 2。→ 提醒學員「要的是 ndarray，不是 list」。
2. **用 `sum(a)/len(a)` 算平均**：技術上可行，但沒學到 `.mean()` 這個 method，教學上要糾正。
3. **篩選寫成 `a[a>25 and a<100]`**：Python 的 `and` 對 ndarray 會 ValueError。必須用 `&` 並加括號：`a[(a>25) & (a<100)]`。

### 核心題常見錯誤

1. **用 for-loop 數數**：例如 `for p in prices: if p>1000: cnt+=1`。違反向量化原則，要求改成 `(prices>1000).sum()`。
2. **argsort 誤解**：把回傳值當成「排序後的數值」直接輸出。要強調它是「索引」，需要再 `stocks[top3_idx]` 才能拿到值。
3. **前 3 名方向搞反**：`sorted_idx[:3]` 拿到的是**最小**的 3 個（因為預設升冪）。正確是 `sorted_idx[-3:][::-1]`。
4. **第 3 小題忘了「乘 50」**：只算了 `cheap_prices.sum()`，漏掉「每個進 50 個」的語意。

### 挑戰題常見錯誤

1. **還在用 for-loop**：題目明確禁止，必須退件要求重寫。可以引導：「如果我給你 100 萬筆，你的 for-loop 會跑幾秒？」
2. **巢狀 where 條件重疊**：例如外層寫 `stocks > 20`、內層又寫 `stocks > 100`，造成邏輯錯亂。要強調「外層 True 分支代表已經確定，內層只處理 False 分支」。
3. **用 `if stocks >= 100:`**：ndarray 不能直接拿來當 `if` 判斷（會 ValueError），必須用 `np.where`。這個錯誤正是「從 Python 思維切換到 NumPy 思維」的最大關卡。
4. **忘了檢查分支完整性**：三個條件要涵蓋所有庫存值，不然會有 NaN 漏網之魚。建議驗證：三個分支的商品數加起來 = 總商品數。

---

## 📚 延伸閱讀

- 官方文件：<https://numpy.org/doc/stable/user/basics.broadcasting.html>
- 向量化效能迷思：for-loop 慢的真正原因是「Python 物件開銷」而非「迴圈本身」
- 下一節 **S2**：Pandas 登場，我們會用「弄髒的訂單 CSV」練習資料清洗

> ### 💡 知識補給站 — When Data Doesn't Fit in Memory
> 
> 本課的 `products.csv` 只有幾十 KB，但真實工作中的 log 檔、交易紀錄可能幾十 GB，遠超電腦記憶體。
> 
> **三個層級的處理策略**：
> - **小資料（< 1 GB）**：直接用 Pandas 讀進來，今天的做法就夠了
> - **中資料（1~10 GB）**：用 `pd.read_csv(path, chunksize=10000)` 分批讀取、逐批累加（S2 會教）
> - **大資料（> 10 GB）**：換引擎 — **Dask** 或 **Polars** 的語法幾乎和 Pandas 一樣，但能處理超過 RAM 的資料
> 
> 類比：你不會把整座圖書館搬回家才開始查資料，你會**一次借幾本** → 這就是 chunked processing 的概念。
> 
> *延伸關鍵字：chunked processing, Dask, Polars, out-of-core computing*